<a href="https://colab.research.google.com/github/panaddatappoomee/food-selection-data-science/blob/main/Luandary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "93934b00",
   "metadata": {},
   "source": [
    "# 🧺 Demo: ฟังก์ชันไพธอนทำงานจริงเมื่อ \"ลูกค้าเดินเข้าร้าน\"\n",
    "### SC 612 104 Essential Data Science — ตัวอย่างสาธิตกระบวนการทำงานของฟังก์ชัน\n",
    "\n",
    "โน้ตบุ๊กนี้เป็น **ตัวอย่างประกอบ** การนำเสนอธุรกิจด้วยโค้ดไพธอน โดยเน้น **ให้เห็นภาพว่าฟังก์ชันแต่ละตัวทำงานยังไงจริง ๆ** ทีละขั้นตอน เหมือนดูลูกค้า\n",
    "  1 คนเดินเข้าร้าน แล้วตามดูว่าโค้ดแต่ละบรรทัด/แต่ละฟังก์ชันถูกเรียกเมื่อไหร่ รับ input อะไร คืนค่าอะไร\n",
    "\n",
    "**ใช้เป็นแนวทาง**: กลุ่มของนักศึกษาสามารถทำ demo แบบนี้กับองค์กรของกลุ่มตัวเอง เพื่อโชว์อาจารย์/เพื่อนร่วมชั้น\n",
    "ตอนนำเสนอว่าโปรแกรมทำงานจริง ไม่ใช่แค่รันแล้วได้ตารางข้อมูลออกมาเฉย ๆ\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "afd90426",
   "metadata": {},
   "source": [
    "## ส่วนที่ 1 — เตรียม class และฟังก์ชัน (เหมือนตัวอย่างร้านซักรีดเดิม)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "0f9f85b2",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.069658Z",
     "iopub.status.busy": "2026-08-15T09:58:46.069470Z",
     "iopub.status.idle": "2026-08-15T09:58:46.074599Z",
     "shell.execute_reply": "2026-08-15T09:58:46.073802Z"
    }
   },
   "outputs": [],
   "source": [
    "import random\n",
    "import time\n",
    "\n",
    "random.seed(7)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "a2b72541",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.076508Z",
     "iopub.status.busy": "2026-08-15T09:58:46.075960Z",
     "iopub.status.idle": "2026-08-15T09:58:46.083034Z",
     "shell.execute_reply": "2026-08-15T09:58:46.082240Z"
    }
   },
   "outputs": [],
   "source": [
    "class Customer:\n",
    "    \"\"\"ลูกค้าของร้านซักรีด\"\"\"\n",
    "\n",
    "    def __init__(self, customer_id, name, is_vip=False):\n",
    "        self.customer_id = customer_id\n",
    "        self.name = name\n",
    "        self.is_vip = is_vip\n",
    "\n",
    "    def get_discount_rate(self):\n",
    "        \"\"\"คืนค่าส่วนลด (สัดส่วน) ตามสถานะสมาชิก VIP\"\"\"\n",
    "        return 0.10 if self.is_vip else 0.0\n",
    "\n",
    "\n",
    "class LaundryOrder:\n",
    "    \"\"\"ออเดอร์ซักผ้า 1 รายการ\"\"\"\n",
    "\n",
    "    PRICE_PER_KG = {\n",
    "        \"ซัก-อบ\": 40,\n",
    "        \"ซัก-อบ-รีด\": 60,\n",
    "        \"รีดอย่างเดียว\": 30,\n",
    "    }\n",
    "\n",
    "    def __init__(self, order_id, customer, weight_kg, service_type):\n",
    "        self.order_id = order_id\n",
    "        self.customer = customer\n",
    "        self.weight_kg = weight_kg\n",
    "        self.service_type = service_type\n",
    "        self.status = \"รอดำเนินการ\"\n",
    "\n",
    "    def calculate_price(self):\n",
    "        base_price = self.weight_kg * self.PRICE_PER_KG[self.service_type]\n",
    "        discount = self.customer.get_discount_rate()\n",
    "        return round(base_price * (1 - discount), 2)\n",
    "\n",
    "    def mark_done(self):\n",
    "        self.status = \"เสร็จแล้ว\""
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "316b49e6",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.084918Z",
     "iopub.status.busy": "2026-08-15T09:58:46.084340Z",
     "iopub.status.idle": "2026-08-15T09:58:46.088927Z",
     "shell.execute_reply": "2026-08-15T09:58:46.088135Z"
    }
   },
   "outputs": [],
   "source": [
    "def generate_thai_name():\n",
    "    \"\"\"ฟังก์ชัน: สุ่มชื่อ-นามสกุลลูกค้า -> คืนค่าเป็น string\"\"\"\n",
    "    first_names = [\"มานี\", \"สมชาย\", \"วิชัย\", \"สมหญิง\", \"ปรีชา\", \"อรุณี\"]\n",
    "    last_names = [\"ใจดี\", \"รักเรียน\", \"สายทอง\", \"ศรีสุข\"]\n",
    "    return f\"{random.choice(first_names)} {random.choice(last_names)}\"\n",
    "\n",
    "\n",
    "def random_weight(min_kg=1.0, max_kg=8.0):\n",
    "    \"\"\"ฟังก์ชัน: สุ่มน้ำหนักผ้า -> คืนค่าเป็น float\n",
    "    มี default argument: เรียกเฉย ๆ ก็ได้ หรือระบุช่วงเองก็ได้\"\"\"\n",
    "    return round(random.uniform(min_kg, max_kg), 1)\n",
    "\n",
    "\n",
    "def format_currency(amount, symbol=\"บาท\"):\n",
    "    \"\"\"ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตริงราคา -> คืนค่าเป็น string\"\"\"\n",
    "    return f\"{amount:,.2f} {symbol}\""
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b9f41ee4",
   "metadata": {},
   "source": [
    "## ส่วนที่ 2 — ทดสอบฟังก์ชันทีละตัว ก่อนเอาไปประกอบเป็นกระบวนการ\n",
    "\n",
    "ก่อนจะเอาฟังก์ชันไปใช้จริง ควรลองเรียกดูเฉย ๆ ทีละตัวก่อน เพื่อดูว่า **input ที่ใส่เข้าไป**\n",
    "กับ **output ที่ได้กลับมา** ตรงกับที่ออกแบบไว้หรือไม่ — นี่คือหลักการพื้นฐานของการเขียนฟังก์ชัน:\n",
    "กำหนด parameter ชัดเจน แล้ว return ค่าที่ใช้ต่อได้\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "44809763",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.090614Z",
     "iopub.status.busy": "2026-08-15T09:58:46.090095Z",
     "iopub.status.idle": "2026-08-15T09:58:46.094501Z",
     "shell.execute_reply": "2026-08-15T09:58:46.093806Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "ชื่อที่สุ่มได้: มานี ใจดี\n",
      "ชื่อที่สุ่มได้: สมชาย ใจดี\n",
      "ชื่อที่สุ่มได้: ปรีชา ศรีสุข\n"
     ]
    }
   ],
   "source": [
    "# เรียก generate_thai_name() 3 ครั้ง -> ทุกครั้งได้ชื่อสุ่มไม่ซ้ำแบบ (แสดงว่าฟังก์ชันทำงานทุกครั้งที่เรียก)\n",
    "for _ in range(3):\n",
    "    print(\"ชื่อที่สุ่มได้:\", generate_thai_name())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "12f4c49a",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.095914Z",
     "iopub.status.busy": "2026-08-15T09:58:46.095770Z",
     "iopub.status.idle": "2026-08-15T09:58:46.100077Z",
     "shell.execute_reply": "2026-08-15T09:58:46.099181Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "เรียกแบบ default (ไม่ใส่ argument): 1.3\n",
      "เรียกแบบระบุช่วงเอง (2.0 - 5.0 กก.): 2.7\n",
      "เรียกแบบระบุแค่ max_kg (keyword argument): 2.1\n"
     ]
    }
   ],
   "source": [
    "# เปรียบเทียบการเรียก random_weight() แบบไม่ระบุค่า (ใช้ default) กับแบบระบุช่วงเอง\n",
    "print(\"เรียกแบบ default (ไม่ใส่ argument):\", random_weight())\n",
    "print(\"เรียกแบบระบุช่วงเอง (2.0 - 5.0 กก.):\", random_weight(2.0, 5.0))\n",
    "print(\"เรียกแบบระบุแค่ max_kg (keyword argument):\", random_weight(max_kg=3.0))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "cc9a8fbf",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.101927Z",
     "iopub.status.busy": "2026-08-15T09:58:46.101363Z",
     "iopub.status.idle": "2026-08-15T09:58:46.105666Z",
     "shell.execute_reply": "2026-08-15T09:58:46.104993Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "ก่อนจัดรูปแบบ: 1234.5 <class 'float'>\n",
      "หลังจัดรูปแบบ: 1,234.50 บาท <class 'str'>\n"
     ]
    }
   ],
   "source": [
    "# ฟังก์ชัน format_currency() รับตัวเลข -> คืนค่าเป็นสตริงที่จัดรูปแบบแล้ว\n",
    "raw_price = 1234.5\n",
    "print(\"ก่อนจัดรูปแบบ:\", raw_price, type(raw_price))\n",
    "print(\"หลังจัดรูปแบบ:\", format_currency(raw_price), type(format_currency(raw_price)))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "63d28864",
   "metadata": {},
   "source": [
    "## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา (ให้เห็นว่าฟังก์ชันเรียกฟังก์ชัน/method อื่นต่อได้)\n",
    "\n",
    "ฟังก์ชัน `explain_price_calculation()` ด้านล่างไม่ได้คำนวณราคาเอง แต่ **เรียก method ของ object**\n",
    "(`order.calculate_price()`, `order.customer.get_discount_rate()`) แล้วนำผลลัพธ์มาพิมพ์อธิบายทีละขั้น —\n",
    "นี่คือตัวอย่างของ \"ฟังก์ชันที่ใช้ผลลัพธ์จากฟังก์ชัน/method อื่นต่อ\"\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "714dab4c",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.107574Z",
     "iopub.status.busy": "2026-08-15T09:58:46.107037Z",
     "iopub.status.idle": "2026-08-15T09:58:46.112236Z",
     "shell.execute_reply": "2026-08-15T09:58:46.111444Z"
    }
   },
   "outputs": [],
   "source": [
    "def explain_price_calculation(order):\n",
    "    \"\"\"รับ object LaundryOrder 1 ตัว -> พิมพ์อธิบายการคำนวณราคาทีละขั้นตอน\"\"\"\n",
    "    rate = order.PRICE_PER_KG[order.service_type]\n",
    "    base_price = order.weight_kg * rate\n",
    "    discount_rate = order.customer.get_discount_rate()\n",
    "    final_price = order.calculate_price()  # เรียก method จริงของ object เพื่อเทียบคำตอบ\n",
    "\n",
    "    print(f\"   ราคาต่อกิโลของบริการ '{order.service_type}' = {rate} บาท/กก.\")\n",
    "    print(f\"   ราคาก่อนหักส่วนลด = {order.weight_kg} กก. x {rate} บาท = {format_currency(base_price)}\")\n",
    "    if discount_rate > 0:\n",
    "        print(f\"   ลูกค้าเป็นสมาชิก VIP -> ได้ส่วนลด {discount_rate*100:.0f}%\")\n",
    "    else:\n",
    "        print(\"   ลูกค้าไม่ใช่สมาชิก VIP -> ไม่มีส่วนลด\")\n",
    "    print(f\"   ราคาสุทธิ = {format_currency(final_price)}\")\n",
    "    return final_price"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4c83c208",
   "metadata": {},
   "source": [
    "## ส่วนที่ 4 — จำลอง \"ลูกค้า 1 คนเดินเข้าร้าน\" แบบ step-by-step\n",
    "\n",
    "ตอนนี้เอาฟังก์ชันทั้งหมดที่มีมาประกอบกันเป็น **1 กระบวนการทางธุรกิจ** ผ่านฟังก์ชันเดียวชื่อ\n",
    "`simulate_customer_visit()` — ฟังก์ชันนี้เรียกฟังก์ชันอื่น ๆ ที่สร้างไว้ก่อนหน้าเรียงกันเป็นลำดับ\n",
    "เหมือนพนักงานหน้าร้านทำงานจริง: ทักทาย → ชั่งผ้า → เลือกบริการ → คำนวณราคา → ออกใบเสร็จ\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 55,
   "id": "63a113b6",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.113874Z",
     "iopub.status.busy": "2026-08-15T09:58:46.113693Z",
     "iopub.status.idle": "2026-08-15T09:58:46.120696Z",
     "shell.execute_reply": "2026-08-15T09:58:46.119871Z"
    }
   },
   "outputs": [],
   "source": [
    "def simulate_customer_visit(order_id, customer, pause=0.0):\n",
    "    \"\"\"จำลองขั้นตอนทั้งหมดตอนลูกค้า 1 คนเข้าร้าน แล้วคืนค่า object LaundryOrder ที่สร้างเสร็จแล้ว\n",
    "    pause = หน่วงเวลาระหว่างขั้นตอน (วินาที) เพื่อให้เห็นจังหวะเวลาตอน demo สด (default ไม่หน่วงเวลา)\"\"\"\n",
    "\n",
    "    print(\"=\" * 60)\n",
    "    print(f\"🚪 ออเดอร์ #{order_id}: ลูกค้า '{customer.name}' เดินเข้ามาที่ร้าน \"\n",
    "          f\"({'สมาชิก VIP' if customer.is_vip else 'ลูกค้าทั่วไป'})\")\n",
    "    time.sleep(pause)\n",
    "\n",
    "    weight = random_weight()  # เรียกฟังก์ชันสุ่มน้ำหนัก\n",
    "    print(f\"⚖️ พนักงานชั่งน้ำหนักผ้า: {weight} กก.\")\n",
    "    time.sleep(pause)\n",
    "\n",
    "    service = random.choice(list(LaundryOrder.PRICE_PER_KG.keys()))\n",
    "    print(f\"🧾 ลูกค้าเลือกบริการ: '{service}'\")\n",
    "    time.sleep(pause)\n",
    "\n",
    "    order = LaundryOrder(order_id, customer, weight, service)  # สร้าง object ออเดอร์จริง\n",
    "    print(\"📦 ระบบสร้างออเดอร์ในระบบเรียบร้อย (สถานะ:\", order.status,\")\")\n",
    "    time.sleep(pause)\n",
    "\n",
    "    print(\"💰 คำนวณราคา:\")\n",
    "    final_price = explain_price_calculation(order)  # เรียกฟังก์ชันอธิบายราคาที่เขียนไว้ก่อนหน้า\n",
    "    time.sleep(pause)\n",
    "\n",
    "    random_time = random.random()  # สุ่มตัวเลข 0.0 - 1.0 เพื่อจำลองเวลาที่พนักงานทำงานเสร็จ\n",
    "    time_unit = 720  # 1 หน่วยเวลา = 720 นาที (12 ชั่วโมง) -> เพื่อให้เห็นผลลัพธ์ชัดเจน\n",
    "    print(f\"⏰ พนักงานเริ่มดำเนินการซักผ้า... ผ่านไป {random_time*time_unit:.0f} นาที ({random_time*time_unit//60:.0f} ชั่วโมง {(random_time*time_unit)%60:.0f} นาที)\")\n",
    "    if random_time < 0.85:  # จำลองว่าออเดอร์ส่วนใหญ่ทำเสร็จก่อนลูกค้ามารับ\n",
    "        order.mark_done()\n",
    "        print(f\"✅ งานเสร็จแล้ว! สถานะออเดอร์เปลี่ยนเป็น: '{order.status}'\")\n",
    "    else:\n",
    "        print(f\"⏳ ยังอยู่ระหว่างดำเนินการ สถานะออเดอร์: '{order.status}'\")\n",
    "\n",
    "    print(f\"🧾 ใบเสร็จ #{order_id}: ลูกค้า {customer.name} | {weight} กก. | {service} | \"\n",
    "          f\"ยอดชำระ {format_currency(final_price)}\")\n",
    "\n",
    "    return order"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 59,
   "id": "220de2bf",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.122476Z",
     "iopub.status.busy": "2026-08-15T09:58:46.121904Z",
     "iopub.status.idle": "2026-08-15T09:58:46.126879Z",
     "shell.execute_reply": "2026-08-15T09:58:46.126123Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "============================================================\n",
      "🚪 ออเดอร์ #1: ลูกค้า 'สมหญิง สายทอง' เดินเข้ามาที่ร้าน (สมาชิก VIP)\n",
      "⚖️ พนักงานชั่งน้ำหนักผ้า: 6.2 กก.\n",
      "🧾 ลูกค้าเลือกบริการ: 'ซัก-อบ-รีด'\n",
      "📦 ระบบสร้างออเดอร์ในระบบเรียบร้อย (สถานะ: รอดำเนินการ )\n",
      "💰 คำนวณราคา:\n",
      "   ราคาต่อกิโลของบริการ 'ซัก-อบ-รีด' = 60 บาท/กก.\n",
      "   ราคาก่อนหักส่วนลด = 6.2 กก. x 60 บาท = 372.00 บาท\n",
      "   ลูกค้าเป็นสมาชิก VIP -> ได้ส่วนลด 10%\n",
      "   ราคาสุทธิ = 334.80 บาท\n",
      "⏰ พนักงานเริ่มดำเนินการซักผ้า... ผ่านไป 648 นาที (10 ชั่วโมง 48 นาที)\n",
      "⏳ ยังอยู่ระหว่างดำเนินการ สถานะออเดอร์: 'รอดำเนินการ'\n",
      "🧾 ใบเสร็จ #1: ลูกค้า สมหญิง สายทอง | 6.2 กก. | ซัก-อบ-รีด | ยอดชำระ 334.80 บาท\n"
     ]
    }
   ],
   "source": [
    "# --- เรียกใช้งานจริงกับลูกค้า 1 คน ---\n",
    "customer_a = Customer(customer_id=1, name=\"สมหญิง สายทอง\", is_vip=True)\n",
    "order_a = simulate_customer_visit(order_id=1, customer=customer_a)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c51bc382",
   "metadata": {},
   "source": [
    "สังเกตว่าตอนเรียก `simulate_customer_visit()` แค่บรรทัดเดียว แต่ **เบื้องหลังมีฟังก์ชันย่อยหลายตัว\n",
    "ถูกเรียกต่อกันเป็นลำดับ** (`random_weight`, `random.choice`, `explain_price_calculation`,\n",
    "`order.calculate_price`, `order.mark_done`) — นี่คือหัวใจของการออกแบบโปรแกรมด้วยฟังก์ชัน:\n",
    "แตกปัญหาใหญ่เป็นฟังก์ชันย่อย ๆ ที่แต่ละตัวทำหน้าที่เดียวชัดเจน แล้วค่อยประกอบกัน"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "58761fb6",
   "metadata": {},
   "source": [
    "## ส่วนที่ 5 — จำลองลูกค้าหลายคนเดินเข้าร้านต่อเนื่องกัน\n",
    "\n",
    "ทีนี้ลองเรียก `simulate_customer_visit()` ซ้ำหลาย ๆ ครั้งผ่าน loop เพื่อดูว่าฟังก์ชันเดียวกัน\n",
    "**ทำงานซ้ำได้ถูกต้องทุกครั้ง** โดยแต่ละครั้งได้ผลลัพธ์ต่างกันไปตามการสุ่ม — นี่คือหลักการเดียวกับ\n",
    "ที่ใช้จำลองข้อมูล 300 รายการในโน้ตบุ๊กหลัก เพียงแต่ตรงนี้ทำแค่ไม่กี่คนเพื่อให้เห็นภาพทีละขั้น\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 62,
   "id": "4dd50311",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.128272Z",
     "iopub.status.busy": "2026-08-15T09:58:46.128136Z",
     "iopub.status.idle": "2026-08-15T09:58:46.135716Z",
     "shell.execute_reply": "2026-08-15T09:58:46.134901Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "============================================================\n",
      "🚪 ออเดอร์ #2: ลูกค้า 'มานี ใจดี' เดินเข้ามาที่ร้าน (ลูกค้าทั่วไป)\n",
      "⚖️ พนักงานชั่งน้ำหนักผ้า: 7.6 กก.\n",
      "🧾 ลูกค้าเลือกบริการ: 'รีดอย่างเดียว'\n",
      "📦 ระบบสร้างออเดอร์ในระบบเรียบร้อย (สถานะ: รอดำเนินการ )\n",
      "💰 คำนวณราคา:\n",
      "   ราคาต่อกิโลของบริการ 'รีดอย่างเดียว' = 30 บาท/กก.\n",
      "   ราคาก่อนหักส่วนลด = 7.6 กก. x 30 บาท = 228.00 บาท\n",
      "   ลูกค้าไม่ใช่สมาชิก VIP -> ไม่มีส่วนลด\n",
      "   ราคาสุทธิ = 228.00 บาท\n",
      "⏰ พนักงานเริ่มดำเนินการซักผ้า... ผ่านไป 18 นาที (0 ชั่วโมง 18 นาที)\n",
      "✅ งานเสร็จแล้ว! สถานะออเดอร์เปลี่ยนเป็น: 'เสร็จแล้ว'\n",
      "🧾 ใบเสร็จ #2: ลูกค้า มานี ใจดี | 7.6 กก. | รีดอย่างเดียว | ยอดชำระ 228.00 บาท\n",
      "============================================================\n",
      "🚪 ออเดอร์ #3: ลูกค้า 'วิชัย ศรีสุข' เดินเข้ามาที่ร้าน (ลูกค้าทั่วไป)\n",
      "⚖️ พนักงานชั่งน้ำหนักผ้า: 7.1 กก.\n",
      "🧾 ลูกค้าเลือกบริการ: 'รีดอย่างเดียว'\n",
      "📦 ระบบสร้างออเดอร์ในระบบเรียบร้อย (สถานะ: รอดำเนินการ )\n",
      "💰 คำนวณราคา:\n",
      "   ราคาต่อกิโลของบริการ 'รีดอย่างเดียว' = 30 บาท/กก.\n",
      "   ราคาก่อนหักส่วนลด = 7.1 กก. x 30 บาท = 213.00 บาท\n",
      "   ลูกค้าไม่ใช่สมาชิก VIP -> ไม่มีส่วนลด\n",
      "   ราคาสุทธิ = 213.00 บาท\n",
      "⏰ พนักงานเริ่มดำเนินการซักผ้า... ผ่านไป 271 นาที (4 ชั่วโมง 31 นาที)\n",
      "✅ งานเสร็จแล้ว! สถานะออเดอร์เปลี่ยนเป็น: 'เสร็จแล้ว'\n",
      "🧾 ใบเสร็จ #3: ลูกค้า วิชัย ศรีสุข | 7.1 กก. | รีดอย่างเดียว | ยอดชำระ 213.00 บาท\n",
      "============================================================\n",
      "🚪 ออเดอร์ #4: ลูกค้า 'ปรีชา แก้วมณี' เดินเข้ามาที่ร้าน (สมาชิก VIP)\n",
      "⚖️ พนักงานชั่งน้ำหนักผ้า: 5.4 กก.\n",
      "🧾 ลูกค้าเลือกบริการ: 'ซัก-อบ-รีด'\n",
      "📦 ระบบสร้างออเดอร์ในระบบเรียบร้อย (สถานะ: รอดำเนินการ )\n",
      "💰 คำนวณราคา:\n",
      "   ราคาต่อกิโลของบริการ 'ซัก-อบ-รีด' = 60 บาท/กก.\n",
      "   ราคาก่อนหักส่วนลด = 5.4 กก. x 60 บาท = 324.00 บาท\n",
      "   ลูกค้าเป็นสมาชิก VIP -> ได้ส่วนลด 10%\n",
      "   ราคาสุทธิ = 291.60 บาท\n",
      "⏰ พนักงานเริ่มดำเนินการซักผ้า... ผ่านไป 434 นาที (7 ชั่วโมง 14 นาที)\n",
      "✅ งานเสร็จแล้ว! สถานะออเดอร์เปลี่ยนเป็น: 'เสร็จแล้ว'\n",
      "🧾 ใบเสร็จ #4: ลูกค้า ปรีชา แก้วมณี | 5.4 กก. | ซัก-อบ-รีด | ยอดชำระ 291.60 บาท\n",
      "============================================================\n",
      "🏁 จบรอบสาธิต — วันนี้มีลูกค้าเข้าร้านทั้งหมด 3 คน (ไม่รวมออเดอร์ #1 ก่อนหน้า)\n"
     ]
    }
   ],
   "source": [
    "walk_in_customers = [\n",
    "    Customer(customer_id=2, name=\"มานี ใจดี\", is_vip=False),\n",
    "    Customer(customer_id=3, name=\"วิชัย ศรีสุข\", is_vip=False),\n",
    "    Customer(customer_id=4, name=\"ปรีชา แก้วมณี\", is_vip=True),\n",
    "]\n",
    "\n",
    "completed_orders = []  # เก็บผลลัพธ์ของทุกออเดอร์ที่ simulate ในรอบนี้\n",
    "\n",
    "for i, cust in enumerate(walk_in_customers, start=2):\n",
    "    order = simulate_customer_visit(order_id=i, customer=cust)\n",
    "    completed_orders.append(order)\n",
    "\n",
    "print(\"=\" * 60)\n",
    "print(f\"🏁 จบรอบสาธิต — วันนี้มีลูกค้าเข้าร้านทั้งหมด {len(completed_orders)} คน (ไม่รวมออเดอร์ #1 ก่อนหน้า)\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f283471e",
   "metadata": {},
   "source": [
    "## ส่วนที่ 6 — สรุปผลจากการ demo (ยืนยันว่าฟังก์ชันคืนค่าถูกต้องและใช้ต่อได้จริง)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 68,
   "id": "a9aa1d67",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-08-15T09:58:46.137049Z",
     "iopub.status.busy": "2026-08-15T09:58:46.136914Z",
     "iopub.status.idle": "2026-08-15T09:58:46.141604Z",
     "shell.execute_reply": "2026-08-15T09:58:46.140886Z"
    }
   },
   "outputs": [],
   "source": [
    "all_demo_orders = [order_a] + completed_orders"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 69,
   "id": "67555757",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>order_id</th>\n",
       "      <th>name</th>\n",
       "      <th>service_type</th>\n",
       "      <th>weight_kg</th>\n",
       "      <th>price</th>\n",
       "      <th>status</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>1</td>\n",
       "      <td>สมหญิง สายทอง</td>\n",
       "      <td>ซัก-อบ-รีด</td>\n",
       "      <td>6.2</td>\n",
       "      <td>334.8</td>\n",
       "      <td>รอดำเนินการ</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>2</td>\n",
       "      <td>มานี ใจดี</td>\n",
       "      <td>รีดอย่างเดียว</td>\n",
       "      <td>7.6</td>\n",
       "      <td>228.0</td>\n",
       "      <td>เสร็จแล้ว</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>3</td>\n",
       "      <td>วิชัย ศรีสุข</td>\n",
       "      <td>รีดอย่างเดียว</td>\n",
       "      <td>7.1</td>\n",
       "      <td>213.0</td>\n",
       "      <td>เสร็จแล้ว</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>4</td>\n",
       "      <td>ปรีชา แก้วมณี</td>\n",
       "      <td>ซัก-อบ-รีด</td>\n",
       "      <td>5.4</td>\n",
       "      <td>291.6</td>\n",
       "      <td>เสร็จแล้ว</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "   order_id           name   service_type  weight_kg  price       status\n",
       "0         1  สมหญิง สายทอง     ซัก-อบ-รีด        6.2  334.8  รอดำเนินการ\n",
       "1         2      มานี ใจดี  รีดอย่างเดียว        7.6  228.0    เสร็จแล้ว\n",
       "2         3   วิชัย ศรีสุข  รีดอย่างเดียว        7.1  213.0    เสร็จแล้ว\n",
       "3         4  ปรีชา แก้วมณี     ซัก-อบ-รีด        5.4  291.6    เสร็จแล้ว"
      ]
     },
     "execution_count": 69,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "import pandas as pd\n",
    "\n",
    "summary_rows = [\n",
    "    {\n",
    "        \"order_id\": o.order_id,\n",
    "        \"name\": o.customer.name,\n",
    "        \"service_type\": o.service_type,\n",
    "        \"weight_kg\": o.weight_kg,\n",
    "        \"price\": o.calculate_price(),\n",
    "        \"status\": o.status,\n",
    "    }\n",
    "    for o in all_demo_orders\n",
    "]\n",
    "\n",
    "demo_summary_df = pd.DataFrame(summary_rows)\n",
    "demo_summary_df "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 70,
   "id": "05c2a869",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\n",
      "ยอดขายรวมจากลูกค้าที่เดินเข้าร้านในรอบ demo นี้: 1,067.40 บาท\n"
     ]
    }
   ],
   "source": [
    "total_today = sum(o.calculate_price() for o in all_demo_orders)\n",
    "print(\"\\nยอดขายรวมจากลูกค้าที่เดินเข้าร้านในรอบ demo นี้:\", format_currency(total_today))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fba06db8",
   "metadata": {},
   "source": [
    "## สรุป demo \n",
    "\n",
    "- แสดงให้เห็นว่าฟังก์ชันแต่ละตัว **ไม่ได้ทำงานลอย ๆ** แต่ถูกเรียกตามลำดับเหตุการณ์จริงของธุรกิจ\n",
    "- แสดงว่า **input → process → output** ของแต่ละฟังก์ชันตรวจสอบได้ทุกขั้นตอน ไม่ใช่กล่องดำ\n",
    "- เวลานำเสนอหน้าชั้น กลุ่มสามารถทำฟังก์ชันแบบ `simulate_customer_visit()` ของธุรกิจตัวเองขึ้นมา 1 ตัว\n",
    "  แล้วรันให้อาจารย์/เพื่อนดู เพื่ออธิบายกระบวนการธุรกิจของแต่ละกลุ่ม และเพื่อพิสูจน์ว่าโค้ดที่เขียนมาทำงานได้จริง ไม่ใช่แค่มีไฟล์ CSV ออกมาเฉย ๆ\n",
    "\n",
    "**ขั้นต่อไป**: เอาแนวคิดฟังก์ชัน `simulate_customer_visit()` นี้ไปปรับใช้กับ loop จำลอง 300 รายการ\n",
    "ในโน้ตบุ๊กหลัก (`laundry_example.ipynb`) ได้เลย — โครงสร้างเดียวกัน แค่ไม่ต้อง print ละเอียดทุกขั้นและไม่หน่วงเวลา"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "base",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}



NameError: name 'null' is not defined